# Problema do Carteiro (Vehicle Routing Problem)

O **Problema do Carteiro** modela um agente que precisa visitar um conjunto de endereços (nós no grafo) e, opcionalmente, retornar ao depósito de origem. É uma versão simplificada do **Problema do Caixeiro Viajante (TSP)** — um problema NP-difícil da otimização combinatória.

## Formulação como Problema de Busca

| Componente | Descrição |
|---|---|
| **Estado** | `PostmanState(current_id, delivered: frozenset)` |
| **Estado inicial** | `PostmanState("Arad", frozenset())` |
| **Teste de meta** | Todos os endereços entregues **e** agente no depósito |
| **Ações** | `GoTo(to_id)` — mover para qualquer vizinho adjacente |
| **Custo** | Peso da aresta (km percorridos) |

## Espaço de Estados

Com 6 endereços de entrega, o estado inclui um `frozenset` de entregas realizadas. O espaço teórico é **|cidades| × 2^|entregas|** = 20 × 64 = **1.280 estados**. A natureza combinatória torna este problema muito mais difícil que o roteamento simples.

## Algoritmos Comparados

| Algoritmo | Custo-ótimo | Expansões (neste caso) |
|---|---|---|
| UCS | Sim | 404 |
| A* (h=MST) | Depende da admissibilidade | 37 |
| A* (h=0) | Sim (= UCS) | 412 |
| DFS | Não | 47 |
| BFS | Por passos, não por custo | 431 |

In [1]:
import os, sys
import networkx as nx
import plotly.graph_objects as go
from typing import Any

## Importações do Projeto

Módulos carregados:
- `Graph` — grafo viário subjacente (rede de estradas)
- `PostmanProblem` — encapsula o grafo, depósito, endereços de entrega e flag `return_to_depot`
- `bfs`, `dfs`, `ucs` — buscas cegas (sem heurística)
- `a_star_search` — busca A* com heurística configurável

`PostmanProblem` também implementa `heuristic_mst` — uma heurística admissível baseada em **Árvore Geradora Mínima (MST)** sobre os endereços ainda não entregues.

In [ ]:
sys.path.append(os.path.join(os.getcwd(), 'structures'))
from structures.graph import Graph
from structures.problems.postman import PostmanProblem
from structures.algorithms.blind_search import bfs, dfs, ucs
from structures.algorithms.heuristic_search import a_star_search

## Construção do Grafo

O mesmo grafo da Romênia (20 cidades, 22 estradas) é reutilizado como rede viária. O carteiro se move pelas estradas reais para alcançar os endereços de entrega — não pode "teletransportar" entre cidades não adjacentes.

O grafo é não-direcionado e ponderado (km), o que garante que `PostmanProblem.successors` e `predecessors` sejam simétricas.

In [3]:
def build_romania_map_graph() -> Graph:
    g = Graph(directed=False)
    edges = [
        ("Arad", "Zerind", 75), ("Arad", "Sibiu", 140), ("Arad", "Timisoara", 118),
        ("Zerind", "Oradea", 71),
        ("Oradea", "Sibiu", 151),
        ("Sibiu", "Fagaras", 99), ("Sibiu", "Rimnicu Vilcea", 80),
        ("Fagaras", "Bucharest", 211),
        ("Rimnicu Vilcea", "Pitesti", 97), ("Rimnicu Vilcea", "Craiova", 146),
        ("Timisoara", "Lugoj", 111),
        ("Lugoj", "Mehadia", 70),
        ("Mehadia", "Drobeta", 75),
        ("Drobeta", "Craiova", 120),
        ("Pitesti", "Bucharest", 101),
        ("Bucharest", "Giurgiu", 90),
        ("Bucharest", "Urziceni", 85),
        ("Urziceni", "Hirsova", 98),
        ("Hirsova", "Eforie", 86),
        ("Urziceni", "Vaslui", 142),
        ("Vaslui", "Iasi", 92),
        ("Iasi", "Neamt", 87)
    ]

    for (u, v, w) in edges:
        g.add_edge(u, v, weight=w)

    return g

## Configuração do Problema

O carteiro parte do depósito **Arad** e deve entregar em 6 cidades:

| Endereço | Conexão com Arad |
|---|---|
| Zerind | Direta (75 km) |
| Oradea | Via Zerind (75+71=146 km) |
| Sibiu | Direta (140 km) |
| Fagaras | Via Sibiu (140+99=239 km) |
| Rimnicu Vilcea | Via Sibiu (140+80=220 km) |
| Timisoara | Direta (118 km) |

`return_to_depot=True` obriga o carteiro a voltar para Arad ao final. O `PostmanState.delivered` é um `frozenset` imutável e hasheável — necessário para usar estados como chaves em `visited`.

In [4]:
g = build_romania_map_graph()
deliveries = ["Zerind", "Oradea", "Sibiu", "Fagaras", "Rimnicu Vilcea", "Timisoara"]

problem = PostmanProblem(g, "Arad", deliveries, True)

## UCS — Busca de Custo Uniforme

O UCS expande os estados em ordem crescente de custo acumulado g(n), garantindo a solução de **menor custo total**.

**Complexidade neste problema:**
- Espaço de estados: ~1.280 (20 cidades × 2^6 subconjuntos de entregas)
- 404 nós expandidos, 466 gerados
- Custo ótimo: **1.031 km**

O custo elevado reflete o fato de que o carteiro precisa percorrer o subconjunto de cidades e ainda retornar ao depósito — semelhante a um mini-TSP.

In [5]:
ucs_res = ucs(problem)

## Resultado do UCS

A solução ótima (1.031 km) segue a rota: Arad → Timisoara → Arad → Zerind → Oradea → Sibiu → Rimnicu Vilcea → Sibiu → Fagaras → Sibiu → Arad.

Note que o carteiro **revisita Sibiu três vezes** — isso é necessário porque Sibiu é o hub de acesso a Fagaras e Rimnicu Vilcea. O UCS aceita esses desvios pois o custo total ainda é mínimo.

In [6]:
ucs_res

SearchResult(found=True, state=PostmanState(current_id='Arad', delivered=frozenset({'Sibiu', 'Fagaras', 'Zerind', 'Rimnicu Vilcea', 'Timisoara', 'Oradea'})), actions=[go_to(Timisoara), go_to(Arad), go_to(Zerind), go_to(Oradea), go_to(Sibiu), go_to(Rimnicu Vilcea), go_to(Sibiu), go_to(Fagaras), go_to(Sibiu), go_to(Arad)], path_cost=1031.0, expanded=404, generated=466, max_frontier=86, elapsed_ms=2.4300099994434277)

## A* com Heurística MST

### Heurística da Árvore Geradora Mínima (MST)

`heuristic_mst` estima o custo restante como:

```
h(s) = custo_MST(endereços_não_entregues)
     + menor_aresta(posição_atual → não_entregues)
     + (se return_to_depot) menor_aresta(não_entregues → depósito)
```

A MST é calculada com o algoritmo de **Kruskal** sobre o subgrafo induzido pelos endereços pendentes. É uma heurística admissível para o TSP: a MST fornece um lower bound para qualquer tour que visite todos os nós do subgrafo.

**Resultado:** solução com custo **1.445 km** em apenas 37 expansões (10× menos que UCS). Porém, a heurística MST usa apenas arestas diretas entre os endereços pendentes, sem considerar que o carteiro pode precisar de caminhos multi-hop — isso pode tornar a heurística **inadmissível** em grafos esparsos, explicando a solução subótima.

In [11]:
a_star_res = a_star_search(problem, problem.heuristic_mst)
a_star_res

SearchResult(found=True, state=PostmanState(current_id='Arad', delivered=frozenset({'Sibiu', 'Fagaras', 'Rimnicu Vilcea', 'Zerind', 'Timisoara', 'Oradea'})), actions=[go_to(Timisoara), go_to(Lugoj), go_to(Mehadia), go_to(Drobeta), go_to(Craiova), go_to(Rimnicu Vilcea), go_to(Pitesti), go_to(Bucharest), go_to(Fagaras), go_to(Sibiu), go_to(Oradea), go_to(Zerind), go_to(Arad)], path_cost=1445.0, expanded=37, generated=55, max_frontier=19, elapsed_ms=0.359040000148525)

## A* com Heurística Nula (h=0)

Quando `h(n) = 0` para todo estado, o A* se torna equivalente ao **UCS** — ordena apenas por g(n). Este é um caso degenerado usado para validação:

- Resultado idêntico ao UCS: custo 1.031 km
- Expansões similares: 412 (vs. 404 do UCS puro)
- A pequena diferença em expansões se deve a diferenças de implementação na fila de prioridade

Este experimento confirma a propriedade fundamental: **A* com h=0 é UCS**.

In [12]:
a_star_res = a_star_search(problem, lambda s: 0.0)
a_star_res

SearchResult(found=True, state=PostmanState(current_id='Arad', delivered=frozenset({'Sibiu', 'Fagaras', 'Zerind', 'Rimnicu Vilcea', 'Timisoara', 'Oradea'})), actions=[go_to(Timisoara), go_to(Arad), go_to(Zerind), go_to(Oradea), go_to(Sibiu), go_to(Rimnicu Vilcea), go_to(Sibiu), go_to(Fagaras), go_to(Sibiu), go_to(Arad)], path_cost=1031.0, expanded=412, generated=466, max_frontier=86, elapsed_ms=2.7846500006489805)

## DFS — Busca em Profundidade

O DFS não é adequado para este problema: encontra a **primeira** solução viável, independentemente do custo.

**Resultado:** custo 2.725 km em 25 ações (vs. 1.031 km ótimo) — **2,6× pior que o ótimo**. O DFS segue o ramo mais profundo primeiro (Timisoara → Lugoj → Mehadia → ...) e retorna apenas quando todas as entregas são feitas, acumulando um percurso muito longo.

**Vantagem:** apenas 47 nós expandidos e fronteira máxima de 14 estados — muito eficiente em memória.

In [9]:
dfs_res = dfs(problem)
dfs_res

SearchResult(found=True, state=PostmanState(current_id='Arad', delivered=frozenset({'Sibiu', 'Fagaras', 'Zerind', 'Rimnicu Vilcea', 'Timisoara', 'Oradea'})), actions=[go_to(Timisoara), go_to(Lugoj), go_to(Mehadia), go_to(Drobeta), go_to(Craiova), go_to(Rimnicu Vilcea), go_to(Craiova), go_to(Drobeta), go_to(Mehadia), go_to(Lugoj), go_to(Timisoara), go_to(Arad), go_to(Zerind), go_to(Oradea), go_to(Sibiu), go_to(Fagaras), go_to(Bucharest), go_to(Pitesti), go_to(Rimnicu Vilcea), go_to(Craiova), go_to(Drobeta), go_to(Mehadia), go_to(Lugoj), go_to(Timisoara), go_to(Arad)], path_cost=2725.0, expanded=47, generated=58, max_frontier=14, elapsed_ms=0.2069100000881008)

## BFS — Busca em Largura

O BFS minimiza o **número de ações** (passos), não o custo. Encontrou uma solução com custo 1.031 km em 10 ações — coincidentemente o mesmo custo que o UCS.

## Comparação Final dos Algoritmos

| Algoritmo | Expansões | Custo (km) | Passos | Ótimo em custo? |
|---|---|---|---|---|
| UCS | 404 | 1.031 | 10 | Sim |
| A* (MST) | 37 | 1.445 | 13 | Não |
| A* (h=0) | 412 | 1.031 | 10 | Sim |
| DFS | 47 | 2.725 | 25 | Não |
| BFS | 431 | 1.031 | 10 | Coincidência |

O problema do carteiro ilustra bem o trade-off entre **exploração exaustiva** (UCS, BFS) e **busca guiada** (A* com heurística), onde uma heurística inadmissível pode ser eficiente mas sacrifica a otimalidade.

In [13]:
bfs_res = bfs(problem)
bfs_res

SearchResult(found=True, state=PostmanState(current_id='Arad', delivered=frozenset({'Sibiu', 'Fagaras', 'Zerind', 'Rimnicu Vilcea', 'Timisoara', 'Oradea'})), actions=[go_to(Zerind), go_to(Oradea), go_to(Sibiu), go_to(Fagaras), go_to(Sibiu), go_to(Rimnicu Vilcea), go_to(Sibiu), go_to(Arad), go_to(Timisoara), go_to(Arad)], path_cost=1031.0, expanded=431, generated=480, max_frontier=89, elapsed_ms=1.8892499992944067)